## Run an end-to-end continous training session

In [ ]:
# custom config dict
custom_config = {
  "execution": {
    "exp_root": "c:/Users/FengWen/Documents_Local/caribou_landcover/experiment",
    "verbosity": 'full'
  },
  "foundation": {
    "grid": {
      "mode": "ref",
      "crs": "EPSG:3161",
      "extent": {
        "filename": "on_3161_30m.tif"
      },
      "tile_specs": {
        "size_row": 256,
        "size_col": 256,
        "overlap_row": 128,
        "overlap_col": 128
      }
    },
    "domains": {
      "files": [
        {
          "name": "ecodistrict.tif",
          "index_base": 1
        },
        {
          "name": "geology.tif",
          "index_base": 1
        }
      ]
    },
    "datablocks": {
      "name": "demo_data",
      "filenames": {
        "dev_image": "image.tif",
        "dev_label": "label_lctype.tif",
        "test_image": "image.tif",
        "test_label": "label_lctype.tif",
        "config": "lctype_test.json"
      }
    }
  },
  "transform": {
    "partition": {
      "val_ratio": 0.1,
      "test_ratio": 0.1
    },
    "scoring": {
      "reward": {
        2: 5.0,
        4: 5.0
      }
    },
    "hydration": {
      "max_skew_rate": 10.0
    }
  },
  "models": {
    "use_body": "unetpp",
    "conditioning": {
      "mode": "hybrid"
    }
  },
  "dataspecs": {
    "domain_ids_name": "ecodistrict",
    "domain_vec_name": "geology"
  },
  "session": {
    "mode": "continuous",
    "engine_exec": {
      "enable_logit_adjust": False,
      "logit_adjust_alpha": 1.0
    },
    "engine_tasks": {
      "excluded_cls": {
        "base": [4, 6]
      }
    },
    "orchestration": {
      "monitor": {
        "track_heads": ["base"],
        "patience": 5,
        "min_delta": 0.005,
      },
      "curriculum": {
          "single": {
              "phases": [
                  {
                      "num_epochs": 10,
                      "active_heads": ["base"]
                  }
              ]
          }
      }
    }
  }
}

In [ ]:
# local import
import landseg.adapters.api as api

# get default configs
default_config = api.get_default_config(pipeline='model-train')
# merge custom configs
running_config = api.add_custom_config(default_config, custom_config)
# run
api.run(running_config)